# **______________   STEAM PROJECT   _______________**

The goal of this project is to conduct a global analysis of the games available on Steam's marketplace in order to better understand the videogames ecosystem and today's trends. We want to understand what factors affect the popularity or sales of a video game to better prepare the release of a new revolutionary videogame. 

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StringType, LongType, DoubleType, ArrayType
import datetime

## **I. PREPARING DATA FOR ANALYSIS**

### **1. Importing dataset**

The dataset is stored in a S3 bucket. We will first extract it and see how it looks like with Databricks.

In [0]:
filepath = 's3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json'

In [0]:
raw_dataset = spark.read.format('json').load(filepath)

### **2. Main characteristics**

#### **2.1. Size of the dataset**

In [0]:
total_nb_rows = raw_dataset.count()
print (f" This dataset has {total_nb_rows} rows and {len(raw_dataset.columns)} columns.")

#### **2.2. Organization of the dataset**

In [0]:
raw_dataset.show(5) # Display the first 5 rows

We can see that this dataset is composed of 2 columns, one for the id (simple value) and the other one for the the data (nested format for each rows). Let's observe one row from the data column : 

In [0]:
raw_dataset.select(F.col("data")).take(1)

To better understand the structure of this nested data, we will analyze the schema of the dataset.

In [0]:
raw_dataset.printSchema() # Analyze how the dataset is structured

When we observe the schema of the dataset we notice that the variables are spread into several levels : 
- LEVEL N°1 : 
    - `data`=> STRUCT
    - `id`
- LEVEL N°2 : 
    - `appid`
    - `categories`=> ARRAY
    - `ccu`
    - `developer`
    - `discount`
    - `genre`
    - `header_image`
    - `initialprice`
    - `languages`
    - `name`
    - `negative`
    - `owners`
    - `platforms` => STRUCT
    - `positive`
    - `price`
    - `publisher`
    - `release_date`
    - `required_age`
    - `short_description`
    - `tags` => STRUCT
    - `type`
    - `website`
- LEVEL N°3 :
    - `element`
    - `linux`
    - `mac`
    - `windows`
    - all the tags
    

From this schema we can observe that : 
- we have one nested list (ARRAY) and 3 nested dictionaries(STRUCT). We will use the "dot notation" for the STRUCT and an "explode" for the array in order to "flatten" this nested file. 
- the information related to prices : `initialprice`, `discount`and `price` are strings. We will have to convert them into Float for further analyses.
- the information related to date `release_date` is a string and will have to be converted as a date format. 

#### **2.3. Get all nested columns**

We need to create a function to get all the nested columns. This function deals with simple columns, StructType and ArrayType. We need to escape special characters in the name of the columns to avoid parsing errors (for example : in the tags like `1980s`).

In [0]:
def get_nested_columns(schema, prefix=""):
    columns = []
    for field in schema.fields:
        # Escape field names
        valid_field_name = f"`{field.name}`"
        name = f"{prefix}.{valid_field_name}" if prefix else valid_field_name
        
        # Recursion if StructType
        if isinstance(field.dataType, StructType):
            columns += get_nested_columns(field.dataType, name)
        else:
            columns.append(name)
    return columns

In [0]:
all_column_names = get_nested_columns(raw_dataset.schema)
all_column_names

### **3. NULL values analysis**

In [0]:
# Count number of nulls for each column and calculate the percentage (we use * to transform the list into arguments for agg())
null_stats = raw_dataset.agg(*[
    F.round((F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / F.lit(total_nb_rows) * 100.0), 2)
    .alias(c.replace(".", "_").replace("`", ""))
    for c in all_column_names
]).collect()[0]

# Transform into a DataFrame
null_df = spark.createDataFrame(
    list(null_stats.asDict().items()),
    ["column_name", "null_percentage"]
).orderBy(F.col("null_percentage").desc())

display(null_df)

From this table we can observe 2 main points : 
- There are no missing values for all the columns that are not tags
- There are high amounts of null values for tags. Here we can deduce that the null values represents games without the specific considered tag and not a missing information so we will replace them by 0 (see part 5.1) and instead of evaluating the percentage of nulls for the tags, we will calculate the frequency.

### **4. Duplicates analysis**

In [0]:
duplicates = (
    raw_dataset
    .groupBy(raw_dataset.columns)
    .count()
    .filter(F.col("count") > 1)
).display()

In [0]:
nb_appid = raw_dataset.select(F.countDistinct(F.col("data.appid"))).collect()[0][0]
nb_names = raw_dataset.select(F.countDistinct(F.col("data.name"))).collect()[0][0]
print("Number of appid:", nb_appid)
print("Number of game names:",nb_names)              
print("Number of games with the same name:",nb_appid - nb_names)            
double_names = raw_dataset.groupBy(F.col("data.name")).count().filter(F.col("count") > 1)

duplicates = (
    raw_dataset
    .join(double_names, raw_dataset["data.name"] == double_names["name"]).orderBy(F.col("data.name"))
)
duplicates.display()

There are no complete duplicated rows but we can observe 261 rows with repeated names but different appid, it can represent several versions of the same game.

### **5. Drop useless columns**

#### **5.1. Drop rare tags**

In [0]:
# Replace NULL by 0 for tags
tag_columns = [c for c in all_column_names if "tags" in c]

df_clean = raw_dataset.fillna(0, subset=tag_columns)

In [0]:
# Analysis of the frequency of each tag (% of games with the tag)
tag_frequencies = {}
    
# % of games with the tag(> 0 votes)
freq_tag_row = df_clean.agg(*[
    F.round(
        F.sum(F.when(F.col(tag) > 0, 1)) / F.lit(total_nb_rows) * 100,
        2
    ).alias(tag.replace("`data`.`tags`.", ""))
    for tag in tag_columns
]).collect()[0]

# Transform into a DataFrame
freq_tag_df = spark.createDataFrame(
    list(freq_tag_row.asDict().items()),
    ["column_name", "frequency_percentage"]
).orderBy(F.col("frequency_percentage").desc())

display(freq_tag_df)

The top 5 tags (present in more than 20% of the games) are : 
- `Indie`
- `Singleplayer`
- `Action`
- `Casual`
- `Adventure`
 
 Moreover, a lot of tags are rare (present in less than 1% of all the games, we can drop them).

In [0]:
# Filter only rare tags (<1%)
threshold = 1
rare_tags_df = freq_tag_df.filter(F.col("frequency_percentage") < threshold)

# Create a list of rare tags
rare_tags = [row["column_name"] for row in rare_tags_df.collect()]

# Create corresponding names
rare_tag_columns = [f"`data`.`tags`.`{tag}`" for tag in rare_tags]

# Drop rare tags
df_clean = df_clean.drop(*rare_tag_columns)

# Stats
stats_df = freq_tag_df.agg(
    F.count("*").alias("nb_tot_tags"),
    F.sum(F.when(F.col("frequency_percentage") < 1, 1).otherwise(0)).alias("nb_rare_tags")
).withColumn(
    "percent_removed", F.round(F.col("nb_rare_tags") / F.col("nb_tot_tags") * 100, 2)
)

stats_df.display()

More than half of the tags are rare (<1%) and have been removed from the dataset.

#### **5.2. Drop non useful columns**

Some columns cannot be used for analysis (`id`,`header_image`).

In [0]:
# List of columns to drop
col_to_drop = ["id","data.header_image"]

# Drop rare tags
df_clean = df_clean.drop(*col_to_drop)

### **6. Prepare a cleaned dataset**

#### **6.1. Flatten the nested structure**

In [0]:
# We reuse the path names of the columns from the get_nested_columns function. We don't select the array "categories" so far, in order not too increase too much the number of rows. 
cols_to_select = [c for c in all_column_names if "categories" not in c]

# Flattenning 
df_flat = raw_dataset.select([
    F.col(c).alias(c.replace(".", "_").replace("`", "")) 
    for c in cols_to_select
])
df_flat.display()

#### **6.2. Formatting dates**

`release_date`is the only column concerning dates. It is a string format so we need to convert it into a date format. First, let's analyze the type of dates in the dataset.

In [0]:
df_flat.select("data_release_date").distinct().show(100, False)

We can observe several date formats : 
- yyyy/MM/dd
- yyyy/MM/d
- yyyy/MM 

We need to clean all the dates before converting the strings into a date format to avoid parsing errors. We will complete the dates without days by adding "01". We also use coalesce to deal with several kind of format.

In [0]:
df_flat_with_dates = (
    df_flat
    .withColumn(
        "cleaned_dates",
        F.when(
            F.col("data_release_date").rlike(r"^\d{4}/\d{2}$"),
            F.concat(F.col("data_release_date"), F.lit("/01"))
        )
        .when(
            F.col("data_release_date")=="",
            F.lit(None)
        )
        .otherwise(F.col("data_release_date"))
    )
    .withColumn(
        "parsed_date",
        F.coalesce(
            F.to_date(F.col("cleaned_dates"), "yyyy/MM/d"),
            F.to_date(F.col("cleaned_dates"), "yyyy/MM/dd"),
            F.to_date(F.col("cleaned_dates"), "yyyy/MM"),
            F.lit(None))
    ))
df_flat_with_dates.display(5)

Now we can extract years and months : 

In [0]:
df_flat_extracted_dates = df_flat_with_dates.withColumn(
    "release_year",
    F.year(F.col("parsed_date"))  # Extract year
).withColumn(
    "release_month",
    F.month(F.col("parsed_date"))  # Extract month
)
df_flat_extracted_dates.display()

#### **6.3. Cleaning types**

After dealing with date formats, we need to correct the wrong types we have in the dataset : 
- Information related to prices should be floats instead of strings (`price`, `initialprice`, `discount`).
- The `required_age`should be an integer instead of a string
To do so, we are using `cast()`.


In [0]:
# Convert the wrong types
df_flat_dates_types =  (
    df_flat_extracted_dates
    .withColumn(
        "data_price",
        F.col("data_price").cast("float")
    ) 
    .withColumn(
        "data_discount",
        F.col("data_discount").cast("float")
    )
    .withColumn(
        "data_initialprice",
        F.col("data_initialprice").cast("float")
    )
    .withColumn(
        "data_required_age",
        F.col("data_required_age").cast("int")
    )
)

In [0]:
# We check that there are no NULL created when the function cast() founds a wrong format.
df_flat_dates_types.select([
    F.count(F.when(F.col("data_price").isNull(), 1)).alias("price_null"),
    F.count(F.when(F.col("data_discount").isNull(), 1)).alias("discount_null"),
]).show()

#### **6.4. Adding relevant features**

Now that our dataset is flattened, that useless columns have been removed, and types corrected, we can now add relevant features for our analysis. The elements that will be necessary to judge the impact of some parameters are the clients'rates. 

We have `positive` and `negative` columns. With this columns, we can calculate several columns : `total_reviews`,`rating_score`. (% of positive reviews).

Then we can also create a boolean column `has_discount`.

In [0]:
# Total of reviews per game
df = df_flat_dates_types.withColumn(
    "total_reviews",
    F.col("data_positive") + F.col("data_negative")
)

# Calculate rating scores
df = df.withColumn(
    "rating_score",
    F.when(F.col("total_reviews") > 0,
         F.col("data_positive") / F.col("total_reviews")
    ).otherwise(None)  # Null if no reviews
)

# Create boolean column
df = df.withColumn(
    "has_discount",
    F.col("data_discount") > 0  # 1 = discount, 0 = no discount
)

In [0]:
# Print schema for verification
df.printSchema()

## **II. DATA ANALYSIS**

To answer business questions to understand the main trends of the video game industry, we will divide the analysis into () main axis : 
- Publications (top and flop publishers, evolution of publications over time, etc.)
- Games (top and flop games, languages, restricted ages, etc.)
- Genres (favorite genres, most lucrative genres)
- Prices (stats, distribution and evolution of evolution of prices)
- Technologies

### **1. Publications analysis**

#### **1.1. Publishers main information**

We can first analyse the main characteristics of publisher to answer the following questions : 
- How many publishers does Steam have? 
- How many publications does every publisher have?
- Which are the best or the worst publishers?
- How the number of publications evolved over time?

In [0]:
# Total number of publishers
total_nb_publishers = df.select(F.col("data_publisher")).distinct().count()
print(f" Total number of publishers : {total_nb_publishers}")

In [0]:
# Number of publications per publisher
nb_publications_per_publisher = (
    df
    .groupBy(F.col("data_publisher"))
    .count()
).orderBy(F.desc("count"))

display(nb_publications_per_publisher)

From this table we can see that : 
- The highest number of publications per publisher is 422 and it has been reached by "Big Fish Games"

- If we consider the number of publications, the top 5 publishers are : 
    - Big Fish Games
    - 8floor
    - SEGA
    - Strategy First
    - Square Enix
- A high number of publishers are low publishers with only one publication. 

The graphic shows us that some publishers have a huge impact on the market while the vast majority is represented by small publishers.

Let's see how many publishers have only one publication.

In [0]:
# Number of publishers with only one publication
last_publishers = (
    nb_publications_per_publisher
    .select("data_publisher")
    .where(F.col('count')==1)
)
last_publishers_nb = last_publishers.count()

print(f"Out of {total_nb_publishers} publishers, {last_publishers_nb} have only one publication. This represents {round(last_publishers_nb/total_nb_publishers*100,2)}% of the total.")

# Visualization
pie_data = spark.createDataFrame(
    [
        ("Single publication", last_publishers_nb),
        ("Multiple publications", total_nb_publishers-last_publishers_nb)
    ],
    ["category", "nb_publishers"]
)

pie_data.display()

In [0]:
# Mean number of releases per publisher
nb_publications = df.count()
avg_publications = round(nb_publications/total_nb_publishers,2)

print(f"On average, publishers have done {avg_publications} publications on Steam.")

#### **1.2. Publishers' reviews**

We have compared publishers in terms of number of publications, let's see now the ones that are best rated. Are there games that were really well rated with only a few publications? Preferring quality over quantity? For that, we will compare ratios of positive rates instead of total numbers of rates. 

In [0]:
reviews_per_publisher = (
    df
    .groupBy("data_publisher")
    .agg(
        F.sum("data_positive").alias("total_positive_reviews"),
        F.sum("data_negative").alias("total_negative_reviews"),
        F.sum("total_reviews").alias("total_reviews_per_publisher"),
        F.round(100*F.avg("rating_score"),2).alias("avg_rating_score (%)"),
        F.count("data_publisher").alias("nb_publications"))
    .orderBy(F.desc(F.col("avg_rating_score (%)"))
    )
)
reviews_per_publisher.display()

We ordered the publishers according to their average rate score. We can observe that a lot of small publishers with only one publication have a high rate score. This is because they have only a few positive reviews and no negative reviews.

We need to conduct a deeper analysis to take into considaration not only the average rate score but also the number of publications so that it better reflects reality and business needs.

To do so, we can use the Bayesian score to weight the positive ratio by the total number of votes, to prevent publishers with very few votes from artificially appearing at the top.

**Formula**: 
$$
bayesian-score = \frac{v}{v+m}R + \frac{m}{v+m}C
$$

Where:

- R = average rating of the publisher  
- v = total number of reviews per publisher 
- m = minimum number of reviews required  
- C = global average rating across all publishers


This formula shrinks scores with few reviews toward the global mean, producing a more reliable ranking.

Publishers with zero reviews were excluded from the ranking, as their ratings contain no information about user perception.  
Since the Bayesian score relies on the number of reviews to estimate reliability, publishers without reviews would simply receive the global average score and therefore provide no meaningful ranking signal.

In [0]:
# Calculate the number of publishers with no reviews
no_review_count = reviews_per_publisher.filter(
    F.col("total_reviews_per_publisher") == 0
).count()

print(f"There are {no_review_count} publishers with no reviews at all (it represents {round(no_review_count/total_nb_publishers*100,2)}% of all publishers).")


We now remove them : 

In [0]:
# Drop publishers with no reviews
reviews_per_publisher = reviews_per_publisher.filter(
    F.col("total_reviews_per_publisher") > 0
)

In [0]:
# Calculate "C" : global average ration across all publishers
global_avg = (
  reviews_per_publisher
  .groupBy()
  .agg(
      F.avg("avg_rating_score (%)").alias("global_mean")
  )
).collect()[0]['global_mean']

print(f"The average rating score is {round(global_avg,2)}%")

Now let's identify "m" as the 80% percentiles of reviews : 

In [0]:
# Calculate "m" (we will fix "m" as the number of reviews at the 80th percentile)
m = (
    reviews_per_publisher
    .select(F.percentile_approx(F.col("total_reviews_per_publisher"), 0.80).alias("m"))
    .collect()[0]['m']
)

print(f"m = {m}")


In [0]:
# Add Bayesian rating score into the DataFrame
reviews_per_publisher = (
    reviews_per_publisher
    .withColumn(
        "bayesian_score",
        (
            (F.col("total_reviews_per_publisher") /
            (F.col("total_reviews_per_publisher") + F.lit(m))) * F.col("avg_rating_score (%)")
            +
            (F.lit(m) /
            (F.col("total_reviews_per_publisher") + F.lit(m))) * F.lit(global_avg)
        )
    )
)

In [0]:
# Top 5 publishers with the highest Bayesian rating score
top_5_bayesian = (
    reviews_per_publisher
    .orderBy(F.desc(F.col("bayesian_score")))
    .limit(5)
    .display()
)


The top 5 publishers according to their reviews are :
- adamgryu
- Igara Studio
- Studio Minus
- poncle
- HIKARI FIELD, NekoNyan Ltd.

They are small publishers with one or two games released, but those games have been positively reviewed by more than 10 000 players. It means that some of the small publishers can have a real positive impact for Steam.

In [0]:
# Bottom 5 publishers with the lowest Bayesian rating score
last_5_bayesian = (
    reviews_per_publisher
    .orderBy(F.asc(F.col("bayesian_score")))
    .limit(5)
    .display()
)

The bottom 5 publishers according to their reviews are :
- Asylum Entertainment Inc.
- Badland Studio
- Dark Day Interactive
- Atari, RCTO Productions
- 22cans Ltd.

They are small publishers as well with one or two games released, but those games have a higher proportion of negative reviews. 

Some small publishers can be really appreciated by the players, others are not, we will understand later what seduces players in a game. 

In [0]:
# Bayesian score of the publishers with the highest number of publications 
(
    reviews_per_publisher
    .orderBy(F.desc(F.col("nb_publications")))
    .limit(5)
    .display()
)

The five main publishers with have volumes of publications that we identified earlier have a Bayesian score above 70% except for "Strategy First" with a score of 56%.

#### **1.3. Time vs publications**

**_1.3.1. Evolution of the number of publications per year_**

Now, let's study the evolution of the publications over time. 
Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
releases_per_year = (
    df
    .groupBy("release_year")
    .count()
    .orderBy("release_year")
)

releases_per_year.display()

We can observe the following trends in the number of game releases:

- **1997–2013:** Slow but steady growth in releases, with the number of releases gradually increasing from 2 to around 500.  
- **2013–2018:** Rapid exponential growth, reaching a peak of more than 7,600 releases in 2018.  
- **2019:** Slight decrease compared to 2018, signaling a slowdown after the exponential growth phase (dropped of about 700 releases).
- **2020-2021:** Recovery with a new peak, likely due to market adjustments after the previous slowdown.  
- **2022:** New decline observed (dropped to 7400 releases), potentially influenced by external factors such as COVID-19 disruptions in 2020, which may have delayed development and publishing schedules.

In [0]:
top_year = (
    releases_per_year
    .select(
        F.col("release_year"),
        F.col("count").alias('nb_releases')
        )
    .orderBy(F.desc(F.col("nb_releases")))
    .limit(1)
)
top_year.display()

The year with the greater amount of release is 2021 with 8823 releases. 

**_1.3.2. Distribution of releases per month_**

Are there month or period of the years with a greater amount of publications? (e.g. before Christmas?).

In [0]:
releases_per_month = (
    df
    .groupBy("release_month")
    .count()
    .orderBy("release_month")
)

releases_per_month.display()

The number of releases over the months is not constant, we can see : 
- 3 pics (March and May with around 4700 releases and a major pic in October (more than 5500 releases))
- the weakest period of the year in term of releases are January, December and June. 

We can suggest that releases are mostly done in October for publishers to boost their sales before Christmas, and that there are less publications in January, December and June because of holidays and a slow down of the business activity. 

#### 1.4. Insights about publications analysis

The publications analysis reveals a highly asymmetric market structure on Steam. Indeed, a few major publishers such as Big Fish Games or SEGA concentrate a large volume of releases, but the vast majority of publishers have only one publication (representing more than half of all publishers). This reflects the democratization of digital distribution, which allows small independent studios to access the market at a low cost. 

Regarding quality, the Bayesian score shows that the best-rated publishers are not necessarily the most prolific ones : small studios like Adamgryu or Poncle, with a single release, outperform established players thanks to highly appreciated games. 

From a temporal perspective, the platform experienced exponential growth in publications between 2013 and 2018, followed by a stabilization with a peak in 2021, suggesting a progressive market saturation. 

Finally, the monthly distribution of releases shows a concentration in October, likely to maximize sales ahead of the holiday season, while January and December remain the quietest periods.

### **2. Games analysis**

#### 2.1 Games' reviews

In [0]:
nb_games = df.count()
print(f"There are {nb_games} games in the dataset.")

As for the publishers, we can calculate the Bayesian score for each game allowing us to ponderate the rate ratio by the number of reviews. 

The formula is the same but this time we have : 
- R : games individual ratings
- v : number of reviews for an individual game
- m : minimum number of reviews required to trust a game's rating fully
- C : global average rating across all games in the dataset

First we need to remove games with no publications at all.

In [0]:
# Calculate the number of games with no reviews
no_review_count = df.filter(
    F.col("total_reviews") == 0
).count()

print(f"There are {no_review_count} game(s) with no reviews at all (it represents {round(no_review_count/nb_games*100,2)}% of all publishers).")


We now remove them : 

In [0]:
# Drop publishers with no reviews
df = df.filter(
    F.col("total_reviews") > 0
)

In [0]:
# Calculate "C":
global_avg_games = df.agg(F.avg("rating_score").alias("global_avg")).collect()[0]["global_avg"]

print(f"The average rating score is {global_avg_games:.2f}")

In [0]:
# Calculate "m" (we will fix "m" as the number of reviews at the 80th percentile)
m = (
    df
    .select(F.percentile_approx(F.col("total_reviews"), 0.80).alias("m"))
    .collect()[0]['m']
)

print(f"m = {m}")


In [0]:
df = df.withColumn(
    "bayesian_score",
    (F.col("total_reviews") / (F.col("total_reviews") + F.lit(m))) * F.col("rating_score") +
    (F.lit(m) / (F.col("total_reviews") + F.lit(m))) * F.lit(global_avg_games)
)

In [0]:
top_5_games = df.orderBy(F.desc("bayesian_score")).select(
    "data_name","data_publisher","bayesian_score", "total_reviews"
).limit(5)
top_5_games.display()

The best games are : 
- People Playground	_(Studio Minus)_
- Aseprite	_(Igara Studio)_
- Portal 2	_(Valve)_
- A Short Hike _(adamgryu)_
- Vampire Survivors	_(poncle)_

We found almost the same results than for the best publishers analysis before, that is understandable because they had only one publication, so they published successful games that we see here. The only difference is "Portal 2" from Valve, that was not in our list of top five publishers. 

In [0]:
reviews_per_publisher.filter(F.col("data_publisher") == "Valve").display()

The publisher Valve has 33 publications and the greatest amount of positive reviews (but a great amount of negatives reviews as well) explaining its bayesian score.

#### 2.2 Games' languages

- How many languages are represented? 
- Are some languages more sold than others? 
- Are some languages present in better rated games?

In [0]:
languages = (
    df
    .select([F.col("data_languages"), F.col("total_reviews"), F.col("bayesian_score")])
            )

languages.display()


Languages are present in the dataset as strings containing one or several languages separated by comas.

In [0]:
# Languages split :
languages_split = languages.withColumn(
        "languages_list",
        F.split(F.col("data_languages"), ", ")
    )

# We now have a list of languages per row that we need to explode : 
languages_exploded = languages_split.select(
    F.explode("languages_list").alias("languages"))

# We can count the number of unique languages
nb_total_languages = languages_exploded.select(
    F.countDistinct("languages").alias("nb_languages")
).collect()[0]['nb_languages']

print(f"There are {nb_total_languages} unique languages in the dataset.") 

In [0]:
# Top 5 languages with the most games : 
languages_count = (
    languages_exploded
    .groupBy("languages")
    .agg(F.count("*").alias("nb_games"))
    .orderBy(F.desc("nb_games"))
    .limit(5)
)

languages_count.display()

We can see the predominance of English for video games. A bit more than half of the games are in English. The other half is divided in an almost balanced manner between the following 4 languages : 
- German
- French
- Russian
- Simplified Chinese.

In [0]:
# Top 5 languages combinaisons: 
languages_list_count = (
    languages_split
    .groupBy("languages_list")
    .agg(F.count("*").alias("nb_games"))
    .orderBy(F.desc("nb_games"))
    .limit(5)
)

languages_list_count.display()

From this graphic we can observe that the top 5 most represented languages and combinaison of languages are : 
- English alone : almost 86% of all games sold 
- English + Russian (5,6%) 
- English + Chinese
- English + Japanese

English is the default language in this industry. More than 80% of games are only available in English.The other languages are mostly present with English as translations.

We can submit the hypothesis that the most popular games (the one with the highest number of rates and positive ratio) are the ones that are translated in more languages. To verify this hypothésis, we can devide games into several categories.

In [0]:
df_lang = (
    languages_split
    .withColumn(
        "nb_languages",
        F.size("languages_list")
    )
    .withColumn(
        "language_group",
        F.when(F.col("nb_languages") <= 2, "1-2")
        .when(F.col("nb_languages") <= 5, "3-5")
        .when(F.col("nb_languages") <= 10, "6-10")
        .otherwise("10+")
    )
)

In [0]:
language_stats = (
    df_lang
    .groupBy("nb_languages")
    .agg(
        F.count("*").alias("nb_games"),
        F.round(F.avg("bayesian_score"),2).alias("avg_bayesian_score"),
        F.expr("percentile(total_reviews, 0.5)").alias("median_reviews")
    )
)

display(language_stats)

#### 2.3. Insights about games analysis

The games analysis highlights that the best-rated titles (measured by the Bayesian score) are often produced by small independent studios with a single publication like : People Playground, Aseprite, Vampire Survivors, confirming the trend observed in the publishers analysis. 

Regarding languages, English is largely dominant : more than 86% of games are available exclusively in English, while the remaining games are mostly translated alongside English, with Russian (5.6%), Chinese and Japanese being the most common additions. 

Interestingly, games available in a greater number of languages tend to attract significantly more reviews and achieve higher Bayesian scores. This suggests that language accessibility is not only a commercial choice but also a quality signal : the most successful games invest in broader localization to reach a wider audience.

### **3. Genres, categories and tags analysis**

#### 3.1. Genres analysis

We can check if there are genres that are more sold than others, that have better reviews, etc.


There are games with several genres so we can split them in order to identify genres that are more represented.

In [0]:
# Genre table
genres_df = (
    df
    .select(F.col("data_name"), F.col("data_genre"), F.col("total_reviews"), F.col("bayesian_score"))
    .withColumn(
        "genres_list",
        F.split(F.col("data_genre"), ", ")
    )
    .withColumn(
        "nb_genres",
        F.size("genres_list")
    )
)

# Genre split
genres_split = genres_df.withColumn(
        "genres_list",
        F.split(F.col("data_genre"), ", ")
    )

# We now have a list of genres per row that we need to explode : 
genres_exploded = genres_split.withColumn("genres",
    F.explode("genres_list").alias("genres"))

# We can count the number of unique genres
nb_total_genres = genres_exploded.select(
    F.countDistinct("genres").alias("nb_genres")
).collect()[0]['nb_genres']

print(f"There are {nb_total_genres} unique genres in the dataset.") 

In [0]:

genres_exploded.display()

Let's identify the most popular genres:

In [0]:
from pyspark.sql.window import Window
# Top 5 genres (higher number of games) : 
genres_count = (
    genres_exploded
    .groupBy("genres")
    .agg(F.countDistinct("data_name").alias("nb_games"))
    .withColumn(
        "%_of_total",
        F.round(
            100*F.col("nb_games") / F.sum("nb_games").over(Window.partitionBy()),
            2
        )
    )
    .orderBy(F.desc("nb_games"))
)

genres_count.display()

The top 5 genres are present in around 75% of all games : 
- `Indie` (1/4 of all games)
- `Action` (around 15%)
- `Casual` (around 14%)
- `Adventure` (around 13%)
- `Strategy` (around 7%)

In [0]:
# Top 5 genres combinaisons: 
genres_list_count = (
    genres_split
    .groupBy("genres_list")
    .agg(F.count("*").alias("nb_games"))
    .orderBy(F.desc("nb_games"))
    .limit(5)
)

genres_list_count.display()

Among all combinaisons of genres, the top 5 are : 
- Indie + Action
- Indie + Casual
- Indie + Action + Adventure
- Indie + Adventure
- Indie + Action + casual

Indie games (independant) games are the favorite genre of players. 

Are these games prefered by players in terms of reviews and rating? 

In [0]:
genre_stats = (
    genres_exploded
    .groupBy("genres")
    .agg(
        F.countDistinct("*").alias("nb_games"),
        F.round(F.avg("bayesian_score"), 2).alias("avg_bayesian_score"),
        F.sum(F.col("total_reviews")).alias("total_reviews"),
    )
    .orderBy(F.desc("total_reviews"))
)

genre_stats.display()


If we consider the toal number of reviews, here are the top 5 genres : 
- `Action`
- `Indie`
- `Adventure`
- `Free to Play`
- `RPG`

Free to play and RPG were not in the top 5 of genres according to the number of games, it means there are present in less games, but games with more reviews. 

On the last graphic, we can see that the average Bayesian score is quite the same (between 0,73 to 0,75 per genre, except for the genre "movie" but this is explained by the fact that there is only one game with this genre). It seems that the selection of the genre is not a determinant factor of the perceived quality. 

#### 3.2. Categories analysis

The "category" column from the initial table was not selected previously for the flattening operation to avoid to increase the number of rows. Now we can explode it and join it to the cleaned dataframe.

In [0]:
categories_exploded = (
    raw_dataset
    .select(
        F.col("data.appid").alias("data_appid"),
        F.explode(F.col("data.categories")).alias("category")
    )
)

# We join the exploded categories with df the cleaned dataset
df_with_categories = df.join(
    categories_exploded,
    on="data_appid",
    how="left"   # Keep all games, even those without categories
)

In [0]:
nb_total_categories = (
    df_with_categories
    .select(F.col("category"))
    .distinct()
    .count()
)

print(f"There are {nb_total_categories} unique categories in the dataset.")

In [0]:
categories_stats = (
    df_with_categories
    .groupBy(F.col("category"))
    .agg(
        F.count("*").alias("nb_games"),
        F.round(F.avg("rating_score"), 2).alias("avg_rating"),
        F.round(F.avg("bayesian_score"), 2).alias("avg_bayesian_score"),
        F.sum(F.col("total_reviews")).alias("total_reviews"),
        F.expr("percentile(total_reviews, 0.5)").alias("median_reviews")
    )
    .withColumn(
        "%_of_games",
        F.round(100*F.col("nb_games") / F.sum("nb_games").over(Window.partitionBy()),2)
        )
    .orderBy(F.desc("nb_games"))
    .limit(5)
)

categories_stats.display()

There are 37 different categories and the 5 main ones are : 

- `Single-player` (27,1%)
- `Steam Achievements` (14,2%)
- `Steam Cloud`(7,4%)
- `Full controller support` (6,2%)
- `Multi-player` (5,96)

All the other categories are represented in less than 5% of all games each.

#### 3.3. Tags analysis

We already know from the part I.5.1 that the top 5 tags (present in more than 20% of the games) are : 
- `Indie`
- `Singleplayer`
- `Action`
- `Casual`
- `Adventure`

and that 228 tags have been removed because they were rare (represented in less than 1% of the games). We can see that the main tags correspond to a mix of the main genre and the main categories.

#### 3.4. Insights about Genre, Category and Tag analysis

The genres, categories and tags analysis paints a consistent picture of the Steam ecosystem. 

In terms of genres, 75% of all games are concentrated in just 5 categories 
— `Indie` (25%), 
- `Action` (15%), 
- `Casual` (14%), 
- `Adventure` (13%) 
- `Strategy` (7%) 

With Indie + Action and Indie + Casual being the most frequent combinations. 

When looking at total reviews rather than number of games, `Free to Play` and `RPG` enter the top 5, indicating that these genres, though less represented, generate strong player engagement. Notably, the average Bayesian score is remarkably stable across all genres (~0.73–0.75), suggesting that genre alone is not a determining factor of perceived quality. 

Regarding categories, `Single-player` is by far the most represented (27.1%), followed by `Steam Achievements` (14.2%) and `Steam Cloud` (7.4%), reflecting players' expectations in terms of features. 

Finally, the top tags (`Indie`, `Singleplayer`, `Action`, `Casual`, `Adventure`) are perfectly consistent with the genres and categories findings, confirming these as the defining characteristics of the average Steam game.

### **4. Prices analysis**

We can investigate what is the effect of prices on the number of reviews : What is the general distribution of prices? Are top games more expensive or cheaper? Are games with a high proportion of negative reviews cheaper? Are big publishers offering more discounts? etc. There are 3 variables giving us information about prices : 
- `discount`
- `initialprice`
- `price`
If there is no discount : `initialprice` = `price`.

#### 4.1. Prices main information

**_4.1.1. General prices stats_**

In [0]:
# Selecting interesting columns
price_info = df.select(
    F.col("data_name"),
    F.col("data_publisher"),
    F.col("data_initialprice"),
    F.col("data_discount"),
    F.col("data_price"),
    F.col("data_positive"),
    F.col("data_negative"),
    F.col("total_reviews"),
    F.col("rating_score"),
    F.col("has_discount"),
    F.col("bayesian_score"),
    F.col("release_year"),
    F.col("release_month")
)
price_info.display()

In [0]:
# Distribution of all prices
prices_distribution = (
    price_info
    .groupBy("data_price")
    .count()
    .orderBy(F.desc("count"))
)
prices_distribution.display()

We can see than more than 700 games have a price = 0, Steam offers free games, the economic model of these game is different from the others so we will split the dataset into 2 categories. 

In [0]:
free_games = price_info.filter(F.col("data_price") == 0)
paid_games = price_info.filter(F.col("data_price") > 0)

In [0]:
# We verify that all free games are initially free and not discounted to free
tot_free = len(price_info.filter(F.col("data_price") == 0).collect())
tot_initial_free = len(price_info.filter(F.col("data_initialprice") == 0).collect())

tot_free == tot_initial_free

Let's compare the ratings and review count between free and paid games.

In [0]:
# Comparison of ratings between free and paid games
compared_ratings = (
    price_info
    .groupBy("has_discount")
    .agg(
        F.round(F.avg("rating_score"),2).alias("avg_rating"),
        F.min("rating_score").alias("min_rating"),
        F.max("rating_score").alias("max_rating"),
        F.round(F.avg("bayesian_score"), 2).alias("avg_bayesian_score"),
        F.count("*").alias("nb_games"),
    )
)
compared_ratings.display()

In [0]:
# Distribution of paid_games only
paid_games_distribution = (
    paid_games
    .groupBy("data_price")
    .count()
    .orderBy(F.desc("count"))
)
paid_games_distribution.display()

The prices follow a long-tail distribution, with most games priced between 28 and 7000 of the currency used for the dataset.

In [0]:
stat_prices_paid_games =(
    paid_games
    .agg(
        F.round(F.avg("data_initialprice"),2).alias("avg_initialprice"),
        F.round(F.avg("data_discount"),2).alias("avg_discount"),
        F.min("data_discount").alias("min_discount"),
        F.max("data_discount").alias("max_discount"),
        F.round(F.avg("data_price"),2).alias("avg_price"),
        F.min("data_price").alias("min_price"),
        F.max("data_price").alias("max_price"),
        F.count("*").alias("nb_games"),
    )
)
stat_prices_paid_games.display()

These statistics are giving us the following information : 
- median price : 599 
- High percentil (99%) : 4999

In [0]:
# Identification of expensive games (>99th percentile)
percentil_99 = paid_games.approxQuantile("data_price", [0.99], 0)[0]
print(f"99th percentile price: {percentil_99}")

# Filter most expensive games
most_expensive_games = (
    price_info
    .filter(F.col("data_price") > percentil_99)
)
most_expensive_games.display()

print(f"There are {most_expensive_games.count()} games with price > {percentil_99} (99th percentile)")

These expensive games must represent collector editions and special offers

Now that we have observed the distribution of prices we are going to segment the paid_games dataframe into several categories using price percentiles to ensure balanced groups across segments despite the highly right-skewed distribution. Here are the segments : 
- 0–25 %	very cheap games
- 25–50 %	cheap games
- 50–75 %	mid-range games
- 75–90 %	expensive games
- 90–100 %	premium games

In [0]:
# Calculate quantiles
quantiles = paid_games.approxQuantile(
    "data_price",
    [0.25, 0.5, 0.75, 0.9],
    0
)

p25, p50, p75, p90 = quantiles

In [0]:
# Segmentation of the prices
paid_games = paid_games.withColumn(
    "price_segment",
    F.when(F.col("data_price") <= p25, "very_low_price")
    .when(F.col("data_price") <= p50, "low_price")
    .when(F.col("data_price") <= p75, "mid_price")
    .when(F.col("data_price") <= p90, "high_price")
    .otherwise("premium_price")
)

In [0]:
# Distribution of games according to the segmentation
paid_games_segmentation = (
    paid_games
    .groupBy("price_segment")
    .count()
    .orderBy(F.desc("count"))
)
paid_games_segmentation.display()

We can observe that more than a quarter of all games have a very low price, almost half of them have a low to mid price and the last quarter have a price from high to premium.

**_4.1.2. Discounts analysis_**

What is the proportion of discounted games?

In [0]:
nb_total = price_info.count()

# Ratio of discounted games
discount_ratio = (
    price_info
    .agg(
        (
            F.sum((F.col("data_discount") != 0).cast("int")) /
            F.count("*")
        ).alias("discount_ratio")
    )
)

ratio = discount_ratio.collect()[0]["discount_ratio"]

print(f"Only {100 * ratio:.2f}% of games are discounted.")

What are the main discounted applied?

In [0]:
# Distribution of discounts
discount_distribution = (
    price_info
    .filter(F.col("data_discount") != 0)
    .groupBy("data_discount")
    .count()
    .orderBy(F.desc("count"))
)
discount_distribution.display()

The main discount percentages applied are tens (50% then 90%,80%) etc. 

Which games are mostly concerned by discounts? Is it the most expensive ones?

In [0]:
# Proportion of discounted games per segment
discount_per_segment = (
    paid_games
    .groupBy("price_segment")
    .agg(
        (
            F.sum((F.col("data_discount") != 0).cast("int")) /
            F.count("*")
        ).alias("%_discounted_games")
    )
)
display(discount_per_segment)

Almost half of the very low prices games are concerned by discounts. Whereas less than 7% of premium prices games are discounted.

#### 4.2. Prices according to publishers

Are prices dependent of the size of the publisher ? Are some publishers offering more discounts than others? Are some publishers offering more free games than others?

In [0]:
stat_price_per_publisher = (
    price_info
    .groupBy("data_publisher")
    .agg( 
        F.round(F.avg("data_initialprice"),2).alias("avg_initialprice"),
        F.round(F.avg("data_discount"),2).alias("avg_discount"),
        F.min("data_discount").alias("min_discount"),
        F.max("data_discount").alias("max_discount"),
        F.round(F.avg("data_price"),2).alias("avg_price"),
        F.min("data_price").alias("min_price"),
        F.max("data_price").alias("max_price"),
        F.count("*").alias("nb_games")
    ))

stat_price_per_publisher.display(5)

#### 4.3. Prices evolution over time

Have the prices evolved over time? 
Are there periods of the year with more discount? 
Are there periods of the year where the games are best sold? 

**_4.3.1. Evolution of prices over years_**

In order to see if the prices evolved over time we will see the evolution of several percentiles : 5% / 50% / 95%. This will allow us to see the principal trends (increase, decrease, rise of cheap or really expensive games, etc.)

In [0]:
price_trend_years = (
    price_info
    .groupBy("release_year")
    .agg(
        F.expr("percentile(data_price, 0.5)").alias("median_price"),
        F.expr("percentile(data_price, 0.05)").alias("p05_price"),
        F.expr("percentile(data_price, 0.95)").alias("p95_price")
    )
    .orderBy("release_year")
)

price_trend_years.display()

We can observe the following trends in prices over time : 

- 1997 to 2005 : Prices are evolving more often during this period of time, than can be explained by the small amount of sales where a different price on a game had a biger impact. The average prices dropped between 1997 to 1999 (from about 1500 to 500) and increased back to the initial price in 2002.
- From 2006 : 
    - p05% : There are almost no more really cheap games (p05) until 2021 where it started to increase a bit again. This can be explained by the fact that the company was growing rapidely and didn't need to offer cheap prices anymore compared to its early stages. 
    - p95% : We also have a stabilization of really high priced games during that period, with pics in 2012, 2013 and 2019. Then after a drop in 2020 a new increased is observed from 2020 to 2022.


**_4.3.2. Evolution of prices over month_**

Are there period of the year with different prices (more discounts, etc.)?

In [0]:
# Distribution of prices over months
price_trend_months = (
    price_info
    .groupBy("release_month")
    .agg(
        F.percentile_approx("data_price", 0.5).alias("median_price"),
        F.percentile_approx("data_price", 0.05).alias("p05_price"),
        F.percentile_approx("data_price", 0.95).alias("p95_price")
    )
    .orderBy("release_month")
)

display(price_trend_months)

#### 4.4. Insights about prices analysis

The prices analysis reveals a highly skewed distribution across all the games offered by Steam. 

Some games are free-to-play games (over 700 games) all of which were originally free rather than discounted. 

Among paid games, the distribution follows a long-tail pattern : the median price stands at 599 (in the dataset's currency unit) while the 99th percentile reaches 4999, indicating a small number of premium or collector editions at much higher price points. 

More than a quarter of paid games fall into the very low price segment, while roughly half are in the low to mid range, leaving only 25% in the high to premium category. 

Discounts are relatively rare and concern only a small fraction of the catalog, with round values dominating (50%, 80% and 90% being the most common). 

Interestingly, discount prevalence is inversely correlated with price : almost half of very low-priced games are discounted, compared to less than 7% of premium games, suggesting that discounts are used as a volume strategy for cheaper titles rather than a positioning tool for high-end ones. 

Finally, the temporal analysis shows that prices were more volatile in Steam's early years, dropping from ~1500 to ~500 between 1997 and 1999 before stabilizing from 2006 onward, with occasional spikes in the 95th percentile in 2012, 2013 and 2019, likely reflecting the emergence of AAA titles and special editions.

### **5. Technologies analysis**

We can now analyze which platform is the most popular.

In [0]:
platforms_df = df.select(
    F.col("data_platforms_windows").alias("windows"),
    F.col("data_platforms_mac").alias("mac"),
    F.col("data_platforms_linux").alias("linux"),
    F.col("rating_score"),
    F.col("total_reviews")
)
display(platforms_df)


In [0]:
platform_stats = df.select(
    F.col("data_platforms_windows").alias("windows"),
    F.col("data_platforms_mac").alias("mac"),
    F.col("data_platforms_linux").alias("linux"),
    F.col("rating_score"),
    F.col("total_reviews")
).groupBy("windows", "mac", "linux").agg(
    F.count("*").alias("nb_games"),
    F.round(100 * F.col("nb_games")/F.lit(df.count()),2).alias("%_of_games"),
    F.round(F.avg("rating_score"),2).alias("avg_rating")
).withColumn("%_of_games", F.round(100 * F.col("nb_games") / F.lit(df.count()), 2)).orderBy("%_of_games", ascending=False)

platform_stats.display()

In [0]:
platforms_df.select(
    F.round(100*F.mean(F.col("windows").cast("int")),2).alias("windows_share"),
    F.round(100*F.mean(F.col("mac").cast("int")),2).alias("mac_share"),
    F.round(100*F.mean(F.col("linux").cast("int")),2).alias("linux_share")
).display()

We can observe that games can be deployed on one or several platforms. The predominant combinaison is Windows alone (74% of the games), whereas Linux or Mac alone represent less than 0,1% of the games. 
We have the following combinaisons : 
- Windows alone (74,1%)
- Windows + Mac + Linux (12,2%)
- Windows + Mac (10,7%)
- Windows + Mac + Linux (< 3%)
- Mac alone (< 0,05%)
- Linux alone(< 0,05%)
- Mac + Linux (< 0,05%)
- Window + Linux (not present)

Windows is used for more than 99,97% of games (alone for multi-platforms games). Windows is the standard platform for PC gaming. Developing without Windows makes almost no commercial sense.

Is there an influence of the platform on the average rating score?

In [0]:
platform_rating = platforms_df.select(
    F.round(100* F.mean(F.when(F.col("windows"), F.col("rating_score"))),2).alias("avg_rating_windows"),
    F.round(100* F.mean(F.when(F.col("mac"), F.col("rating_score"))),2).alias("avg_rating_mac"),
    F.round(100* F.mean(F.when(F.col("linux"), F.col("rating_score"))),2).alias("avg_rating_linux")
)

display(platform_rating)

It seems that games supporting Mac or Linux tend to have slightly higher average ratings than Windows games. However, this likely reflects a selection effect: games released on multiple platforms are often larger or more successful titles that later receive additional platform support. Moreover, Windows support more than 99% of the games so the good and bad ones. 

#### 5.2. Insights from technology analysis


The technologies analysis clearly establishes Windows as the unchallenged standard for PC gaming on Steam. Windows is present in 99.97% of all games : either exclusively (74.1%) or in combination with other platforms, making it a non-negotiable target for any game release. 

Mac and Linux, on the other hand, are almost never supported as standalone platforms (each representing less than 0.05% of the catalog) and appear primarily as additions to Windows support, with Windows + Mac +Linux (12.2%) and Windows + Mac (10.7%) being the most common multi-platform combinations. 

While games supporting Mac or Linux tend to display slightly higher average ratings, this should not be interpreted as a causal relationship : it most likely reflects a selection effect, whereby larger and more successful titles (which naturally attract better reviews) are the ones that can afford to invest in additional platform ports. 

Developing without Windows support therefore makes almost no commercial sense on Steam.


## **III. CONCLUSIONS**

### 1. Key findings

This analysis of the Steam catalog provides a comprehensive overview of the video game ecosystem and its main trends. 

The market is characterized by a highly asymmetric structure, dominated by a large number of small independent publishers, most of which have only one publication. 

Despite this fragmentation, quality can emerge from anywhere : the best-rated games are often produced by small studios with a single release, proving that a focused approach can outperform volume strategies. 

Indie games represent the largest genre on the platform (25% of all games), and the most successful titles tend to combine Indie with Action or Casual mechanics. 

Across all genres, the average Bayesian score remains remarkably stable (~0.73–0.75), indicating that genre alone does not determine perceived quality. 

Language accessibility plays an important role : games available in more languages attract significantly more reviews and achieve higher scores, with English being the absolute standard (86% of games are English-only). 

From a pricing perspective, the catalog follows a long-tail distribution with a median price of 599, and discounts are primarily used as a volume strategy for lower-priced titles. 

Finally, Windows is the undisputed platform for PC gaming, present in 99.97% of all games, while multi-platform support appears as a feature of already successful titles rather than a driver of success.

### 2. Main recommandations for a new game release

Based on the key findings of this analysis, the following recommendations can be made for the release of a new game on Steam :

- **Prioritize quality over quantity**. The data shows that the best-rated publishers and games are often small studios with a single, highly polished release. Investing in one well-crafted game is a more effective strategy than flooding the market with multiple low-quality titles.

- **Target the Indie+Action or Indie+Casual combination**. These are the most represented and reviewed genre combinations on Steam, indicating strong and established player demand.

- **Release in October**. The monthly analysis clearly shows that October is the peak month for releases, maximizing visibility ahead of the holiday season and benefiting from higher consumer spending in November and December.

- **Ensure English support as a baseline, and invest in localization**. English is non-negotiable (86% of games are English-only), but adding Russian, Chinese or Japanese can significantly expand the potential audience and has been correlated with higher review counts and Bayesian scores.

- **Develop for Windows first**. With 99.97% of Steam games supporting Windows, it is the only platform that cannot be overlooked. Multi-platform support (Mac, Linux) can be considered as a later investment once the game has proven its success.

- **Adopt a competitive pricing strategy**. With more than 50% of paid games priced in the low to mid range, and a median price of 599, pricing a new release too high risks limiting its reach. A mid-range price with a well-timed promotional discount can help drive initial traction.

### 3. Limitations of the dataset

While this analysis provides valuable insights into the Steam ecosystem, several limitations should be taken into account when interpreting the results :

- **No sales data**. The dataset does not include actual sales figures or revenue data. The number of reviews has been used as a proxy for popularity, but it only captures players who actively left a review, which may not be representative of the full player base.

- **Temporal coverage**. The dataset reflects the catalog at a specific point in time and does not capture the full historical evolution of prices, reviews or releases. Some trends observed, particularly for early years (1997–2005), are based on very few games and should be interpreted with caution.

- **Currency ambiguity**. Prices are expressed in a unit that is not explicitly identified in the dataset, making it difficult to draw precise conclusions about absolute price levels or to compare them with market benchmarks.

- **Tag and genre self-reporting**. Genres and tags are assigned by publishers themselves, which can lead to inconsistencies or strategic inflation of certain categories (e.g., labeling a game as "Indie" for visibility purposes).

- **Review bias**. The rating system only reflects the opinion of players who chose to leave a review, which may skew results towards more engaged, and potentially more polarized users.